# NeutralMotion Schedule Tutorial

This notebook shows how to run `naive_dag` and `naive_n_dag` to produce schedule files from QASM circuits or a `step_order` text file. It covers installation, direct Python function calls, and CLI equivalents. 

## Installation

You can use the project from inside a local clone or install it directly from GitHub.

From inside the repository root:

```bash
pip install -e .
pip install matplotlib pylatexenc pandas
```

Directly from GitHub:

```bash
pip install git+https://github.com/Gerwinlab/NeutralMotion.git
```

If you have not installed the package yet but you are running this notebook from the repository root, the next cell adds `src/` to `sys.path` so the imports still work.

In [1]:
from pathlib import Path
import json
import sys

repo_root = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "src" / "naive_n_dag").is_dir())  # works from Examples/ or the repo root
src_dir = repo_root / "src"
if src_dir.exists() and str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from naive_dag.main import main as naive_dag_main
from naive_n_dag.main import load_config as load_naive_n_config
from naive_n_dag.main import main as naive_n_dag_main

In [2]:
qasm_dir = repo_root / "inputs" / "qasm_files" / "qft"
config_dir = repo_root / "benchmarks" / "configs" / "qft"
output_dir = repo_root / "Examples" / "OUTPUT"  # everything this notebook writes goes here
output_dir.mkdir(parents=True, exist_ok=True)

example_qasm = qasm_dir / "qft_6.qasm"
example_qasm.relative_to(repo_root)

PosixPath('inputs/qasm_files/qft/qft_6.qasm')

## Running `naive_dag`

`naive_dag.main(...)` expects:

- a config mapping
- a QASM filename or path
- an optional schedule output directory
- optional `quiet` and `seed` arguments

`naive_dag` reads only the device and timing keys of a config (it always places atoms with its own naive fill), so any solver config works. The first example below reuses `benchmarks/configs/qft/qft_06_n_fastsa.json`, sized for `qft_6.qasm`, and the second shows how to define a small config inline. `naive_dag` names its schedule `<qasm stem>.schedule.txt`, so each run writes to its own folder.

In [3]:
naive_dag_config_path = config_dir / "qft_06_n_fastsa.json"
naive_dag_config = json.loads(naive_dag_config_path.read_text())

naive_dag_exit_code = naive_dag_main(
    naive_dag_config,
    str(example_qasm),
    schedule_output_dir=output_dir / "naive_dag_json_config",
    quiet=False,
    seed=0,
)

naive_dag_schedule = output_dir / "naive_dag_json_config" / "qft_6.schedule.txt"
print("exit_code:", naive_dag_exit_code)
print("schedule:", naive_dag_schedule.relative_to(repo_root))

naive_dag config loaded
dimensions=[3, 3] num_NA=6 qasm_dir=../../../inputs/qasm_files/qft
qasm_file=/home/gage/Quantum_Class/Phys765/inputs/qasm_files/qft/qft_6.qasm
transpile=off
selected_schedule=relocation
candidate_previous_home_time=8847.146619907313 microsecond
candidate_relocation_time=7711.428726381592 microsecond
7711.428726381592 microsecond
schedule_file=/home/gage/Quantum_Class/Phys765/Examples/OUTPUT/naive_dag_json_config/qft_6.schedule.txt
exit_code: 0
schedule: Examples/OUTPUT/naive_dag_json_config/qft_6.schedule.txt


### Inline `naive_dag` config example

If you want a minimal self-contained example, you can also define the config directly in Python.

In [4]:
naive_dag_inline_config = {
    "dimensions": [3, 3],
    "num_NA": 6,
    "qasm_base_dir": str(qasm_dir),
    "transfer_SLM_AOD": "15 microseconds",
    "max_acceleration": "2750 m/s^2",
    "max_velocity": "2 m/s",
    "lattice_spacing": "15 micrometers",
    "average_single_gate_time": "1 microseconds",
    "average_two_gate_time": "0.27 microseconds",
    "t_switch": "1.5 microseconds",
    "movement_shape": "Cubic_Spline",
}

naive_dag_inline_exit_code = naive_dag_main(
    naive_dag_inline_config,
    "qft_6.qasm",
    schedule_output_dir=output_dir / "naive_dag_inline_config",
    quiet=False,
    seed=1,
)

print("exit_code:", naive_dag_inline_exit_code)
print("schedule:", (output_dir / "naive_dag_inline_config" / "qft_6.schedule.txt").relative_to(repo_root))

naive_dag config loaded
dimensions=[3, 3] num_NA=6 qasm_dir=/home/gage/Quantum_Class/Phys765/inputs/qasm_files/qft
qasm_file=/home/gage/Quantum_Class/Phys765/inputs/qasm_files/qft/qft_6.qasm
transpile=off
selected_schedule=relocation
candidate_previous_home_time=7515.796728679925 microsecond
candidate_relocation_time=6367.3154328497885 microsecond
6367.3154328497885 microsecond
schedule_file=/home/gage/Quantum_Class/Phys765/Examples/OUTPUT/naive_dag_inline_config/qft_6.schedule.txt
exit_code: 0
schedule: Examples/OUTPUT/naive_dag_inline_config/qft_6.schedule.txt


In [5]:
print("\n".join(naive_dag_schedule.read_text().splitlines()[:20]))

qasm_file: qft_6.qasm
final_time: 7711.428726381592 microsecond
lattice_spacing: 14.999999999999998 micrometer
random number seed: 0

T=0
initialize q[0] -> (4,0); q[1] -> (4,4); q[2] -> (0,0); q[3] -> (0,4); q[4] -> (2,2); q[5] -> (2,0);
T=1
h q[5];
T=2
load q[5] -> (2,0) : (2,1)
T=3
cp(pi/2) q[5],q[4];
T=4
move q[5] (2,1) : (1,1)
T=5
move q[5] (1,1) : (1,4)
T=6
cp(pi/4) q[5],q[3];
T=7


## Running `naive_n_dag`

`naive_n_dag` has example configs in `benchmarks/configs/`. This version supports extra options such as:

- `output_name` to control the filename
- `log=True` to write a FastSA CSV log when `fill_strategy` is `fastsa`
- `quiet=True` to suppress status printing
- `seed` to make random placement reproducible when the fill strategy uses randomness
- `max_dimension` in config sets the AOD size; atoms always move together when it allows (use `[1, 1]` to move one atom at a time)
- `transpile_optimization_level` / `transpile_basis_gates` in config run the circuit through Qiskit's `transpile` first (default: the circuit as written); both solvers support them
- `ancilla_registers` (with `prefer_moving_ancillas`) names the QASM registers whose qubits are ancillas, so data-ancilla gates move the ancilla
- `step_order` in config to load a `.txt` timestep file (when present, the QASM argument is ignored)

SWAPs in the QASM are scheduled as ordinary two-qubit gates; removing unnecessary ones (for example the final bit reversal of `qft_6.qasm`) is up to the user, by editing the QASM or transpiling at level 2.


In [6]:
naive_n_config_path = config_dir / "qft_06_n_fastsa.json"
naive_n_config = load_naive_n_config(naive_n_config_path)

naive_n_exit_code = naive_n_dag_main(
    naive_n_config,
    "qft_6.qasm",
    schedule_output_dir=output_dir,
    config_path=naive_n_config_path,
    quiet=False,
    seed=0,
    output_name="qft_06_n_fastsa_tutorial",
    log=True,
)

naive_n_schedule = output_dir / "qft_06_n_fastsa_tutorial.schedule.txt"
naive_n_log = output_dir / "qft_06_n_fastsa_tutorial.fastsa_log.csv"
print("exit_code:", naive_n_exit_code)
print("schedule:", naive_n_schedule.relative_to(repo_root))
print("fastsa_log:", naive_n_log.relative_to(repo_root))

naive_n_dag config loaded
dimensions=[3, 3] num_NA=6 qasm_dir=/home/gage/Quantum_Class/Phys765/inputs/qasm_files/qft
qasm_file=/home/gage/Quantum_Class/Phys765/inputs/qasm_files/qft/qft_6.qasm
max_dimension=[3, 3]
fill_strategy=fastsa
schedule_output_dir=/home/gage/Quantum_Class/Phys765/Examples/OUTPUT
reorder_cz=True
transpile=off
displacement_precompile_layers=18
selected_schedule=reuse
reused_groups=4
tsp_solves=3 tsp_enumerated=3 tsp_gadget=0 tsp_exact_wins=0 tsp_alignment_wins=0 tsp_exact_failures=0 tsp_repaired=0 tsp_repaired_used=0 tsp_seconds_saved=0us
candidate_reuse_time=7550.4950669135915 microsecond
candidate_individual_time=7550.4950669135915 microsecond
two_qubit_layers=18
scheduled_timesteps=70
estimated_time=7550.4950669135915 microsecond
schedule_file=/home/gage/Quantum_Class/Phys765/Examples/OUTPUT/qft_06_n_fastsa_tutorial.schedule.txt
fastsa_log_file=/home/gage/Quantum_Class/Phys765/Examples/OUTPUT/qft_06_n_fastsa_tutorial.fastsa_log.csv
exit_code: 0
schedule: Exampl

In [7]:
for path in [naive_n_schedule, naive_n_log]:
    print(f"{path.name}: {'present' if path.exists() else 'missing'}")

qft_06_n_fastsa_tutorial.schedule.txt: present
qft_06_n_fastsa_tutorial.fastsa_log.csv: present


## Using `step_order` Text Input

You can provide a text-based two-qubit gate order with the config key `step_order` (an absolute path, or a path relative to the JSON config file).

Behavior when `step_order` is provided:

- the QASM argument is ignored (warning emitted)
- only two-qubit gates are loaded from the text file (warning emitted)
- `single_layers` is returned as an array of empty layers sized to the two-qubit DAG depth (`layers + 1`)
- options that need the full circuit (`reorder_cz`, transpile, `ancilla_registers`) must be off

Expected per-timestep format:

```text
<T>, <gate_index>: <q0> <q1>
... (one or more gate lines for the same T)
T = <T>, cx = <count_of_gate_lines>
```

* this only supports cx gates. The primary focus of this feature is for testing layer creation where each time step becomes a layer.

The next cell writes such a file from the two-qubit layers of `qft_6.qasm` to `Examples/OUTPUT/qft_6_step_order.txt` and schedules it.

In [8]:
from naive_n_dag.dag_helper import load_qasm_to_two_qubit_dag_with_single_qubit_context

two_qubit_dag, _ = load_qasm_to_two_qubit_dag_with_single_qubit_context(example_qasm)
lines, gate_index = [], 0
for t, layer in enumerate(two_qubit_dag.layers(), start=1):
    pairs = [[two_qubit_dag.find_bit(q).index for q in node.qargs] for node in layer["graph"].op_nodes()
             if node.op.name != "barrier"]
    if not pairs:
        continue
    for q0, q1 in pairs:
        lines.append(f"{t}, {gate_index}: {q0} {q1}")
        gate_index += 1
    lines.append(f"T = {t}, cx = {len(pairs)}")
step_order_path = output_dir / "qft_6_step_order.txt"
step_order_path.write_text("\n".join(lines) + "\n")
print("step_order:", step_order_path.relative_to(repo_root))
print("\n".join(lines[:12]))

step_order_config = load_naive_n_config(config_dir / "qft_06_n_deterministic.json")
step_order_config.update(step_order=str(step_order_path), reorder_cz=False)
step_order_exit_code = naive_n_dag_main(
    step_order_config,
    str(example_qasm),  # required by the signature, ignored with step_order
    schedule_output_dir=output_dir,
    quiet=True,
    output_name="qft_6_step_order_tutorial",
)
print("exit_code:", step_order_exit_code)
print("schedule:", (output_dir / "qft_6_step_order_tutorial.schedule.txt").relative_to(repo_root))

step_order: Examples/OUTPUT/qft_6_step_order.txt
1, 0: 5 4
T = 1, cx = 1
2, 1: 5 3
T = 2, cx = 1
3, 2: 5 2
3, 3: 4 3
T = 3, cx = 2
4, 4: 5 1
4, 5: 4 2
T = 4, cx = 2
5, 6: 5 0
5, 7: 4 1
exit_code: 0
schedule: Examples/OUTPUT/qft_6_step_order_tutorial.schedule.txt


/tmp/ipykernel_313700/3996885602.py:21: UserWarning: step_order is set; the provided qasm_file argument will be ignored.
  step_order_exit_code = naive_n_dag_main(
/tmp/ipykernel_313700/3996885602.py:21: UserWarning: step_order input only contains two-qubit gates; single-qubit context is not loaded.
  step_order_exit_code = naive_n_dag_main(


Example command with this mode, using a config that sets `step_order` (and `reorder_cz: false`):

```bash
python -m naive_n_dag path/to/config_with_step_order.json inputs/qasm_files/qft/qft_6.qasm Examples/OUTPUT
```

The QASM argument is still required by the CLI signature, but it is ignored when `step_order` is present in config.


## CLI equivalents

If you prefer the command line, the same runs look like this (from the repository root).

```bash
python -m naive_dag benchmarks/configs/qft/qft_06_n_fastsa.json inputs/qasm_files/qft/qft_6.qasm Examples/OUTPUT/naive_dag_json_config --seed 0
```

```bash
python -m naive_n_dag benchmarks/configs/qft/qft_06_n_fastsa.json qft_6.qasm Examples/OUTPUT --seed 0 --output-name qft_06_n_fastsa_tutorial --log
```

Useful flags:

- `--quiet` suppresses non-error output
- `--dump-config` prints the resolved config and exits
- `--seed` sets the placement seed
- `--output-name` is available in `naive_n_dag`
- `--log` is available in `naive_n_dag` when using FastSA

Config notes:

- Grouped movement is always on; the AOD size `max_dimension` limits it (the old `parallel` key is rejected)
- when `step_order` is present in config, the CLI qasm argument is ignored


# Schedule Output Example

Below is the start of `Examples/OUTPUT/qft_06_n_fastsa_tutorial.schedule.txt`, written by the `naive_n_dag` run above. This is only a small excerpt, not the full file.

```text
solver: naive_n_dag
qasm_file: qft_6.qasm
final_time: 7550.4950669135915 microsecond
lattice_spacing: 14.999999999999998 micrometer
rydberg_radius: 15 micrometer
random number seed: 0

T=0
initialize q[0] -> (2,0); q[1] -> (4,0); q[2] -> (0,2); q[3] -> (0,0); q[4] -> (4,2); q[5] -> (2,2);
T=1
h q[5];
T=2
load q[5] -> (2,2) : (3,2)
T=3
cp(pi/2) q[5],q[4];
T=4
move q[5] (3,2) : (3,1)
T=5
move q[5] (3,1) : (0,1)
T=6
cp(pi/4) q[5],q[3];
```

## What the Header Means

- `solver` is the scheduler that wrote the file (`naive_n_dag` only; `naive_dag` files start at `qasm_file`).
- `qasm_file` is the input source recorded in the schedule header. With `step_order`, this will be the `.txt` step-order filename.
- `qubit_registers` (only for QASM files with several quantum registers) records how their qubits were numbered: `l=q[0:63] r=q[63:126] ...`.
- `transpile` (only when the transpile options are set) records the Qiskit `optimization_level` and `basis_gates`.
- `final_time` is the total estimated execution time of the full schedule.
- `lattice_spacing` is the SLM trap pitch used by the movement model (one grid step is half of it).
- `rydberg_radius` is the fixed 15 micrometer blockade radius; simultaneous two-qubit gates whose atoms are closer than this run in separate pulses.
- `random number seed` is the seed used for randomized placement when a random fill strategy is selected.

## What the Actions Mean

- `initialize` gives the home position of each atom in the lattice. This is where atoms start; `naive_n_dag` returns every atom there after transport, while `naive_dag` may leave an atom in a different trap.
- `load` means an atom is transferred from the Spatial Light Modulator (SLM) into the 2D Acousto-Optic Deflector (AOD) transport system.
- The `load` time includes the SLM-to-AOD transfer time plus the motion required to enter the transport "highway" under the scheduler's timing model.
- `unload` is the reverse operation: the atom leaves the AOD transport path and returns to an SLM trap.
- `move q[5] (3,2) : (3,1)` means atom `q[5]` moves from grid coordinate `(3,2)` to `(3,1)`.
- The user can infer the geometric distance from those coordinates, but the scheduler computes time internally from its movement profile (`movement_shape`) together with the configured `lattice_spacing`.
- Gate lines such as `h q[5];`, `cp(pi/2) q[5],q[4];` and `measure q[0] -> meas[0];` are written as OpenQASM statements.

## Gate Support

Currently, this scheduler supports only 1-qubit and 2-qubit gates. SWAPs are scheduled as ordinary two-qubit gates (the end of this file runs the three bit-reversal SWAPs of `qft_6.qasm`).
